<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/TwitterSentimentPretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install transformers

In [55]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel,AutoTokenizer
from torch.utils.data import DataLoader,Dataset
import os
from google.colab import userdata
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [36]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [37]:
!kaggle datasets download -d suchintikasarkar/sentiment-analysis-for-mental-health

Dataset URL: https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health
License(s): DbCL-1.0
sentiment-analysis-for-mental-health.zip: Skipping, found more recently modified local copy (use --force to force download)


In [38]:
!unzip -q sentiment-analysis-for-mental-health.zip -d ./datafolder/

replace ./datafolder/Combined Data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [39]:
df = pd.read_csv('/content/datafolder/Combined Data.csv')

In [40]:
df.isnull().sum()

,0
Unnamed: 0,0
statement,362
status,0


In [41]:
df = df.dropna(subset=['statement'])

In [42]:
df = df[df['statement'].str.strip() != ""]

In [43]:
statement_text = df['statement'].astype(str).values
status_text = df['status'].astype(str).values

In [44]:
label_encode = LabelEncoder()
status_text_encoded = label_encode.fit_transform(status_text)

In [45]:
len(label_encode.classes_)

7

In [46]:
autoToken = AutoTokenizer.from_pretrained('bert-base-uncased')

In [47]:
max_len = 256
vocab_size = autoToken.vocab_size
num_classes = len(label_encode.classes_)

In [48]:
x_train,x_test,y_train,y_test = train_test_split(statement_text,status_text_encoded,test_size=0.3,random_state=42,stratify=status_text_encoded)

In [50]:
train_encoding = autoToken(text=list(x_train),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
val_encoding = autoToken(text=list(x_test),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class SentimentDataSet(Dataset):
  def __init__(self,encoding,labels,max_len):
    self.encoding = encoding
    self.labels = labels

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    return {
        'input_ids':self.encoding['input_ids'][idx],
        'attention_mask':self.encoding['attention_mask'][idx],
        'labels':torch.tensor(self.labels[idx],dtype=torch.long)
    }

In [ ]:
train_Data = SentimentDataSet(text=train_encoding,labels=y_train,max_len=max_len)
val_Data = SentimentDataSet(text=val_encoding,labels=y_test,max_len=max_len)

In [ ]:
train_ds = DataLoader(dataset=train_Data,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
val_ds = DataLoader(dataset=val_Data,batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

In [52]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
print("GPU Count:", torch.cuda.device_count())

Device: cpu
GPU Count: 0


In [53]:
class BERTPreTrained(nn.Module):
  def __init__(self,num_classes) -> None:
    super().__init__()
    self.bert = AutoModel.from_pretrained('bert-base-uncased')
    self.dropout = nn.Dropout(0.2)
    self.classifier = nn.LazyLinear(num_classes)

  def forward(self,input_ids,attnetionmask):
    output = self.bert(input_ids=input_ids,attention_mask=attnetionmask)

    x = output.last_hidden_state

    mask = attnetionmask.unsqueeze(-1)
    x = (x * mask).sum(dim=1)/mask.sum(dim=1)

    return self.classifier(self.dropout(x))


In [ ]:
model = BERTPreTrained(num_classes=num_classes)
# 1. First, freeze EVERYTHING
for param in model.bert.parameters():
    param.requires_grad = False

# 2. Now, unfreeze ONLY the last 2 layers (Layer 10 and 11)
# And also the 'pooler' which is at the very end
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

for param in model.bert.pooler.parameters():
    param.requires_grad = True

In [ ]:
# MULTI GPU SUPPORT
# -----------------------------------
if torch.cuda.device_count() > 1:
    print("Using Multiple GPUs")
    model = nn.DataParallel(model)

# Move FINAL model to device
model = model.to(device)

In [ ]:
optimizer = optim.Adam(params=model.parameters(),lr=2e-4,weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
epochs = 5

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_Correct = 0
  train_total = 0
  progress_bar_train = tqdm(train_ds, desc=f"Epoch {epoch+1}")

  for batch in progress_bar_train:
    text = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

    output = model(input_ids=text,attnetionmask=attention_mask)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()
    train_loss +=loss.item()
    progress_bar_train.set_postfix(
      loss = loss.item()
    )
    pred = torch.argmax(output,1)
    train_Correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_Correct / train_total
  train_losses = train_loss / len(train_ds)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0
  progress_bar_val = tqdm(val_ds, desc='Validation')
  with torch.no_grad():
    for batch in progress_bar_val:
      text = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      output = model(input_ids=text,attnetionmask=attention_mask)
      loss = loss_fn(output,labels)
      val_loss += loss.item()
      progress_bar_val.set_postfix(
        loss = loss.item()
    )
      pred = torch.argmax(output,1)
      val_correct += (pred == labels).sum().item()
      val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_ds)

    print(f'\nEpochs: {epoch+1}/{epochs}...')
    print(f'Train_acc: {train_accuracy} | Train_loss: {train_losses}')
    print(f'Val_acc: {val_accuracy} | Val_loss: {val_losses}')
    print('==============================================================')


Epochs: 1/5...
Train_acc: 0.7728875149148497 | Train_loss: 0.5873565639838486
Val_acc: 0.7734894020879468 | Val_loss: 0.5729744323955374
